# 02 — Raw Table Schema Inventory and Raw-Row Preview (IEEE-CIS Fraud Detection)

**Project:** Fraud Detection Under Limited Review Capacity: A Temporally Validated Study of Gradient-Boosting Models and Decision Policies
**Programme / module:** MSc Data Science, University of Greenwich — COMP1885 Individual Project
**GitHub Issue:** #44
**Notebook status:** data-understanding evidence. No model is trained, no statistic involving the target is computed, and no Test-period label is read.

## Purpose

[Notebook 01](01_dataset_feasibility_audit.ipynb) established shapes, data types (dtypes), keys, duplicates and missingness for each raw table. This notebook completes the data-understanding evidence for the **two raw source tables considered separately**, before any merge or preparation pipeline is designed:

- **Part A — Schema inventory** (outputs committed): file integrity, the column inventory of each table, the location of the target `isFraud`, the columns shared by both tables, and an exact check of the joined schema against [`config/features.yaml`](../config/features.yaml). Part A reads **column headers only**.
- **Part B — Raw-row preview** (local inspection only): the first and last 10 rows of each table, so the raw values can be seen.

## Test-sealing and data-use rules

- The Test period is relative days 155–182 ([DR-001](../docs/decisions/DR-001-temporal-split-strategy.md)). Its labels are **sealed**: the target column is never read for Test-period rows.
- A relative day is `TransactionDT` (time offset in seconds from an undisclosed reference point) divided by 86,400 seconds and rounded down.
- No prevalence, fraud count or feature–target relationship is computed anywhere in this notebook.
- Raw files stay local in `data/raw/` and are never committed. Part B cells carry the tag `local-only`; their output tables **must be removed before commit**, so that no row-level competition data appear in the repository (Kaggle Competition Rules §7B; [`docs/dataset.md`](../docs/dataset.md)). Column names and schema information are public and may be committed.

Evidence tags: **[D]** observed dataset evidence; **[I]** inference or recommendation.

## 1. Setup

In [1]:
import hashlib
import platform
import sys
from pathlib import Path

import pandas as pd
import yaml
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
TX_PATH = RAW / "train_transaction.csv"
ID_PATH = RAW / "train_identity.csv"
FEATURES_PATH = ROOT / "config" / "features.yaml"

KEY = "TransactionID"
TARGET = "isFraud"
TIME = "TransactionDT"
SECONDS_PER_DAY = 86_400
N_PREVIEW = 10

# File integrity anchors recorded in notebook 01, section 2.1.
EXPECTED_SHA256 = {
    "train_transaction.csv": "3a5c83ab6b3cc13dcabe5ffa9f522307fd5f7f7b6e6f6a60c32284ca6283d642",
    "train_identity.csv": "b63c725d8377be90a995268d97f347c17d456b95db45807adcf9f59cd603c37c",
}

# Chronological partitions by relative day (DR-001).
PARTITIONS = {"Train": (1, 105), "V1": (106, 126), "V2": (127, 154), "Test": (155, 182)}
EXPECTED_PARTITION_ROWS = {"Train": 368_632, "V1": 65_544, "V2": 77_822, "Test": 78_542}


def relative_day(transaction_dt):
    """Relative day = floor(TransactionDT / 86,400); no calendar date is inferred."""
    return transaction_dt // SECONDS_PER_DAY


def partition_of(day):
    for name, (first, last) in PARTITIONS.items():
        if first <= day <= last:
            return name
    raise ValueError(f"Relative day {day} lies outside DR-001 partitions")


def feature_group(col):
    """Organiser feature group of a raw column (column names only; no values)."""
    if col == KEY:
        return "Key"
    if col == TARGET:
        return "Target"
    if col == TIME:
        return "Time"
    if col == "TransactionAmt":
        return "Amount"
    if col == "ProductCD":
        return "Product"
    if col.startswith("card"):
        return "Card"
    if col in ("addr1", "addr2"):
        return "Address"
    if col in ("dist1", "dist2"):
        return "Distance"
    if col.endswith("emaildomain"):
        return "Email domain"
    if col[0] == "C" and col[1:].isdigit():
        return "C (counts)"
    if col[0] == "D" and col[1:].isdigit():
        return "D (time deltas)"
    if col[0] == "M" and col[1:].isdigit():
        return "M (match flags)"
    if col[0] == "V" and col[1:].isdigit():
        return "V (organiser-engineered)"
    if col.startswith("id_"):
        return "Identity id_01-id_38"
    if col in ("DeviceType", "DeviceInfo"):
        return "Device"
    return "Other"


GROUP_ORDER = ["Key", "Target", "Time", "Amount", "Product", "Card", "Address", "Distance",
               "Email domain", "C (counts)", "D (time deltas)", "M (match flags)",
               "V (organiser-engineered)", "Identity id_01-id_38", "Device", "Other"]

print("Python", sys.version.split()[0], "|", platform.platform())
print("pandas", pd.__version__, "| PyYAML", yaml.__version__)
print("Repository root:", ROOT)

Python 3.12.3 | Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39
pandas 3.0.6 | PyYAML 6.0.3
Repository root: /home/chathuranga/projects/fraud-detection-review-capacity


## 2. Part A — Schema inventory (header-only; outputs committed)

Part A reads file bytes (for checksums) and **column headers only** (`nrows=0`). No data row and no label value is loaded.

### 2.1 File integrity

The Secure Hash Algorithm 256-bit (SHA-256) digest of each raw file is compared with the value recorded in notebook 01 §2.1, so both notebooks provably describe the same files.

In [2]:
def sha256(path, chunk=1 << 22):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while block := handle.read(chunk):
            digest.update(block)
    return digest.hexdigest()


integrity_rows = []
for path in (TX_PATH, ID_PATH):
    assert path.exists(), f"Missing raw file: {path}"
    observed = sha256(path)
    integrity_rows.append({
        "file": path.name,
        "size_MB": round(path.stat().st_size / 1e6, 1),
        "sha256": observed,
        "matches notebook 01": observed == EXPECTED_SHA256[path.name],
    })
integrity = pd.DataFrame(integrity_rows)
assert integrity["matches notebook 01"].all(), "Raw file differs from the file audited in notebook 01"
integrity

,file,size_MB,sha256,matches notebook 01
0,train_transaction.csv,683.4,3a5c83ab6b3cc13dcabe5ffa9f522307fd5f7f7b6e6f6a60c32284ca6283d642,True
1,train_identity.csv,26.5,b63c725d8377be90a995268d97f347c17d456b95db45807adcf9f59cd603c37c,True


### 2.2 Column headers of each table

In [3]:
tx_cols = list(pd.read_csv(TX_PATH, nrows=0).columns)
id_cols = list(pd.read_csv(ID_PATH, nrows=0).columns)

for name, cols in (("train_transaction.csv", tx_cols), ("train_identity.csv", id_cols)):
    assert len(cols) == len(set(cols)), f"Duplicate column names in {name}"

assert len(tx_cols) == 394, len(tx_cols)
assert len(id_cols) == 41, len(id_cols)

pd.DataFrame({
    "table": ["train_transaction.csv", "train_identity.csv"],
    "columns (incl. key)": [len(tx_cols), len(id_cols)],
    "first column": [tx_cols[0], id_cols[0]],
    "last column": [tx_cols[-1], id_cols[-1]],
})

,table,columns (incl. key),first column,last column
0,train_transaction.csv,394,TransactionID,V339
1,train_identity.csv,41,TransactionID,DeviceInfo


### 2.3 Column inventory by feature group

Columns are grouped by the organiser's feature groups. "Columns" lists each group's names in file order; three or more consecutively numbered names are shown as a range (for example `C1–C14`).

In [4]:
import re


def compress_names(names):
    """Show runs of 3+ consecutively numbered names with the same prefix as 'first–last'."""
    parts, run = [], []

    def flush():
        if len(run) >= 3:
            parts.append(f"{run[0][2]}–{run[-1][2]}")
        else:
            parts.extend(item[2] for item in run)
        run.clear()

    for name in names:
        match = re.fullmatch(r"(.*?)(\d+)", name)
        if match is None:
            flush()
            parts.append(name)
            continue
        prefix, number = match.group(1), int(match.group(2))
        if run and run[-1][0] == prefix and run[-1][1] == number - 1:
            run.append((prefix, number, name))
        else:
            flush()
            run.append((prefix, number, name))
    flush()
    return ", ".join(parts)


def inventory(cols, table):
    frame = pd.DataFrame({"column": cols})
    frame["group"] = frame["column"].map(feature_group)
    summary = (frame.groupby("group", sort=False)["column"]
               .agg(n_columns="size", columns=compress_names))
    summary = summary.reindex([g for g in GROUP_ORDER if g in summary.index])
    summary.insert(0, "table", table)
    return summary


tx_inventory = inventory(tx_cols, "train_transaction.csv")
id_inventory = inventory(id_cols, "train_identity.csv")
assert "Other" not in tx_inventory.index and "Other" not in id_inventory.index, "Unclassified column found"

with pd.option_context("display.max_colwidth", 400):
    display(tx_inventory)
    display(id_inventory)

,table,n_columns,columns
group,,,
Key,train_transaction.csv,1,TransactionID
Target,train_transaction.csv,1,isFraud
Time,train_transaction.csv,1,TransactionDT
Amount,train_transaction.csv,1,TransactionAmt
Product,train_transaction.csv,1,ProductCD
Card,train_transaction.csv,6,card1–card6
Address,train_transaction.csv,2,"addr1, addr2"
Distance,train_transaction.csv,2,"dist1, dist2"
Email domain,train_transaction.csv,2,"P_emaildomain, R_emaildomain"


,table,n_columns,columns
group,,,
Key,train_identity.csv,1,TransactionID
Identity id_01-id_38,train_identity.csv,38,id_01–id_38
Device,train_identity.csv,2,"DeviceType, DeviceInfo"


### 2.4 Target location and table overlap

In [5]:
shared = sorted(set(tx_cols) & set(id_cols))
target_check = pd.DataFrame({
    "check": [
        "isFraud present in train_transaction.csv",
        "isFraud present in train_identity.csv",
        "TransactionDT present in train_identity.csv",
        "Columns shared by both tables",
    ],
    "result": [TARGET in tx_cols, TARGET in id_cols, TIME in id_cols, ", ".join(shared)],
})
assert TARGET in tx_cols and TARGET not in id_cols
assert shared == [KEY], shared
target_check

,check,result
0,isFraud present in train_transaction.csv,True
1,isFraud present in train_identity.csv,False
2,TransactionDT present in train_identity.csv,False
3,Columns shared by both tables,TransactionID


### 2.5 Joined schema versus `config/features.yaml`

The left join of the transaction table with the identity table on `TransactionID` keeps every transaction row and adds the identity columns ([DR-008](../docs/decisions/DR-008-feature-declaration-and-model-input-policy.md)). The joined column count is:

$$
N_{\mathrm{joined}} = N_{\mathrm{tx}} + N_{\mathrm{id}} - 1 = 394 + 41 - 1 = 434
$$

where:

- `N_joined` = number of columns after the join;
- `N_tx`, `N_id` = numbers of columns in `train_transaction.csv` and `train_identity.csv`;
- the `− 1` removes the duplicated key `TransactionID`.

Each joined column must have **exactly one** declared role in `config/features.yaml`: target (1), excluded (2), categorical (49) or numeric (382).

In [6]:
features = yaml.safe_load(FEATURES_PATH.read_text())
joined_cols = tx_cols + [c for c in id_cols if c != KEY]
assert len(joined_cols) == len(set(joined_cols)) == 434

roles = {
    "target": [features["target"]],
    "excluded": [entry["name"] for entry in features["excluded"]],
    "categorical": features["categorical"]["transaction"] + features["categorical"]["identity"],
    "numeric": features["numeric"]["columns"],
}
declared = [col for cols in roles.values() for col in cols]
assert len(declared) == len(set(declared)), "A column is declared in more than one role"
assert set(features["categorical"]["numeric_coded"]) <= set(roles["categorical"])

missing_from_yaml = sorted(set(joined_cols) - set(declared))
not_in_data = sorted(set(declared) - set(joined_cols))
assert not missing_from_yaml and not not_in_data, (missing_from_yaml, not_in_data)
assert len(declared) == features["source"]["joined_column_count"] == 434
assert len(roles["categorical"]) + len(roles["numeric"]) == features["source"]["model_input_column_count"] == 431

pd.DataFrame({
    "role": list(roles),
    "declared columns": [len(v) for v in roles.values()],
    "expected": [1, 2, 49, 382],
}).assign(match=lambda d: d["declared columns"] == d["expected"])

,role,declared columns,expected,match
0,target,1,1,True
1,excluded,2,2,True
2,categorical,49,49,True
3,numeric,382,382,True


### 2.6 Part A findings

- **[D]** Both raw files match the SHA-256 digests audited in notebook 01 §2.1 (`train_transaction.csv` 683.4 MB; `train_identity.csv` 26.5 MB).
- **[D]** `train_transaction.csv` has 394 columns (first `TransactionID`, last `V339`); `train_identity.csv` has 41 (first `TransactionID`, last `DeviceInfo`). No duplicate column names.
- **[D]** `isFraud` exists only in `train_transaction.csv`. `train_identity.csv` has no target and no `TransactionDT`, so identity rows can be placed in time only through `TransactionID`.
- **[D]** The only column shared by both tables is `TransactionID`.
- **[D]** All 434 joined columns have exactly one role in `config/features.yaml`: 1 target, 2 excluded, 49 categorical, 382 numeric (431 model inputs).
- **[D]** Partition row counts from the label-free lookup (Section 3.1) match DR-001: Train 368,632; V1 65,544; V2 77,822; Test 78,542.

## 3. Part B — Raw-row preview (local inspection only)

> **Before committing:** remove every output of the cells tagged `local-only` (Section 4). They display row-level competition data.

### 3.1 Partition lookup (label-free)

Only `TransactionID` and `TransactionDT` are read here, so each row's relative day and partition can be determined. Identity rows have no `TransactionDT`; their partition is found through this lookup, not from file order. This cell prints only partition row counts, which are compared with DR-001.

In [7]:
keys = pd.read_csv(TX_PATH, usecols=[KEY, TIME])
keys["relative_day"] = relative_day(keys[TIME])
keys["partition"] = keys["relative_day"].map(partition_of)
lookup = keys.set_index(KEY)[["relative_day", "partition"]]
assert lookup.index.is_unique

partition_rows = keys["partition"].value_counts().reindex(list(PARTITIONS))
assert partition_rows.to_dict() == EXPECTED_PARTITION_ROWS, partition_rows.to_dict()
assert int(partition_rows.sum()) == 590_540
partition_rows.rename("rows").to_frame()

,rows
partition,
Train,368632
V1,65544
V2,77822
Test,78542


### 3.2 First 10 rows — `train_transaction.csv` (Train period; `isFraud` shown)

The first rows belong to relative day 1 (Train), where labels are development data. Rows are shown transposed (one column per transaction).

In [ ]:
tx_head = pd.read_csv(TX_PATH, nrows=N_PREVIEW)
tx_head_days = relative_day(tx_head[TIME]).map(partition_of)
assert (tx_head_days == "Train").all(), "First rows are not in the Train period"

with pd.option_context("display.max_rows", 500):
    display(tx_head.T)

### 3.3 First 10 rows — `train_identity.csv`

In [ ]:
id_head = pd.read_csv(ID_PATH, nrows=N_PREVIEW)
id_head_partitions = lookup.loc[id_head[KEY], ["relative_day", "partition"]].reset_index(drop=True)
print("Partitions of these identity rows:", sorted(id_head_partitions["partition"].unique()))

with pd.option_context("display.max_rows", 500):
    display(pd.concat([id_head_partitions, id_head], axis=1).T)

### 3.4 Last 10 rows — `train_transaction.csv` (Test period; `isFraud` never read)

The file is read in chunks **with `isFraud` excluded through `usecols`**, so the Test label column is never loaded. Only the last 10 rows are kept.

In [ ]:
tx_tail = None
for chunk in pd.read_csv(TX_PATH, usecols=lambda col: col != TARGET, chunksize=100_000):
    tx_tail = chunk.tail(N_PREVIEW) if tx_tail is None else pd.concat([tx_tail, chunk]).tail(N_PREVIEW)

assert TARGET not in tx_tail.columns
tx_tail_partitions = relative_day(tx_tail[TIME]).map(partition_of)
print("Partitions of these transaction rows:", sorted(tx_tail_partitions.unique()))

with pd.option_context("display.max_rows", 500):
    display(tx_tail.T)

### 3.5 Last 10 rows — `train_identity.csv`

The identity table has no label column. Partitions come from the `TransactionID` lookup in Section 3.1.

In [ ]:
id_tail = pd.read_csv(ID_PATH).tail(N_PREVIEW)
id_tail_partitions = lookup.loc[id_tail[KEY], ["relative_day", "partition"]].reset_index(drop=True)
print("Partitions of these identity rows:", sorted(id_tail_partitions["partition"].unique()))

with pd.option_context("display.max_rows", 500):
    display(pd.concat([id_tail_partitions, id_tail.reset_index(drop=True)], axis=1).T)

## 4. Before committing

Remove the outputs of all `local-only` cells, keep Part A outputs, then save:

```bash
.venv/bin/jupyter nbconvert --to notebook --inplace \
  --TagRemovePreprocessor.enabled=True \
  --TagRemovePreprocessor.remove_all_outputs_tags local-only \
  notebooks/02_raw_table_schema_inventory.ipynb
```

Checklist:

- [ ] No row-level values remain in any output (Sections 3.2–3.5 are empty).
- [ ] Part A outputs (Sections 2.1–2.5) and the partition counts (Section 3.1) are kept.
- [ ] Notebook 01 is unchanged.

## Symbols and notation

| Symbol / abbreviation | Meaning |
|---|---|
| [D], [I] | Evidence tags: observed dataset evidence; inference or recommendation |
| DR-NNN | Decision Record NNN in `docs/decisions/` |
| dtype | Stored data type of a column |
| `isFraud` | Target: 1 = fraud, 0 = legitimate (in `train_transaction.csv` only) |
| N_joined | Number of columns after the left join (434) |
| N_tx, N_id | Numbers of columns in the transaction (394) and identity (41) tables |
| Relative day | `TransactionDT` divided by 86,400 seconds, rounded down; no calendar date inferred |
| SHA-256 | Secure Hash Algorithm, 256-bit (file checksum) |
| Test | Relative days 155–182; labels sealed |
| Train, V1, V2 | Relative days 1–105; 106–126 (Validation 1); 127–154 (Validation 2) |
| `TransactionDT` | Time offset in seconds from an undisclosed reference point |
| `TransactionID` | Row identifier and join key shared by both tables |